# NeuroLens — Brain Tumor MRI Classification: Baseline Comparison

This notebook compares two ways of building a 4-class brain tumor MRI classifier (glioma, meningioma, no tumor, pituitary):

| Track | Models | Runs |
|---|---|---|
| **A — Fine-tuned pretrained models** | MobileNetV2, EfficientNetB0, Xception | S1 (train new head, backbone frozen) → S2 (also fine-tune the top 20 backbone layers) |
| **B — Trained from scratch** | Custom CNN, Fibonacci-Net (Roy et al. 2025) | Custom CNN: default + hyperparameter-tuned · Fibonacci-Net: 3 learning rates · Fibonacci-Net Ablation A: 1 run |

**How to use this notebook**
- Run it on Google Colab with a GPU (*Runtime → Change runtime type → GPU*).
- Run sections **1–4** first (setup, data, preprocessing, evaluation helpers). After that, each model section can be run **on its own, one at a time**. Every model is created and trained in its own cells — no model shares training code with another.
- Every result is appended to `results.csv` on Google Drive and every trained model is saved there too. If Colab disconnects, re-run sections 1–4 and continue with the model you were on — finished models do not need to be retrained.
- The **validation** set is used for early stopping and for choosing the best configuration; the **test** set is only used to report final performance.

## 1. Setup

### 1.1 Imports

In [ ]:
import os
import json
import time
import random
import hashlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     BatchNormalization, GlobalAveragePooling2D, GlobalMaxPooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, Xception

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             classification_report, confusion_matrix)

sns.set_style("whitegrid")
print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

### 1.2 Configuration

All training settings are defined once here so that every model is trained under the same conditions.

| Setting | Value | Used for |
|---|---|---|
| `IMG_SIZE` | 224 × 224 | All models (standard input size for every architecture) |
| `BATCH_SIZE` | 32 | All models (lower to 16 if the GPU runs out of memory) |
| `EPOCHS_S1` | 15 | S1 — training the new head on a frozen backbone |
| `EPOCHS_FT` | 10 | S2 — fine-tuning (small learning rate, few epochs needed) |
| `EPOCHS_SCRATCH` | 40 | Custom CNN and Fibonacci-Net (trained from random weights, need more epochs) |
| `FT_TOP_LAYERS` | 20 | S2 — number of backbone layers unfrozen |

Early stopping (on validation loss) ends training sooner whenever the model stops improving.

In [ ]:
SEED = 42
IMG_SIZE = (224, 224)
IMG_SHAPE = (224, 224, 3)
BATCH_SIZE = 32
VAL_FRACTION = 0.15

EPOCHS_S1 = 15
EPOCHS_FT = 10
EPOCHS_SCRATCH = 40
FT_TOP_LAYERS = 20

def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds()

### 1.3 Google Drive (results and saved models)

Trained models and the results table are saved to Google Drive so they survive a Colab disconnect.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/NeuroLens"
except ImportError:          # running outside Colab
    OUTPUT_DIR = "outputs"

MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
RESULTS_CSV = os.path.join(OUTPUT_DIR, "results.csv")
os.makedirs(MODEL_DIR, exist_ok=True)
print("Saving models to:", MODEL_DIR)
print("Results table:", RESULTS_CSV)

## 2. Data

### 2.1 Download the dataset (version 1)

The Kaggle *Brain Tumor MRI Dataset* (Nickparvar) combines the Figshare, SARTAJ and Br35H collections. **Version 1** (the original release) is used on purpose: its classes are **imbalanced**, which reflects real clinical data, where tumor types are rarely equally common. (Version 2 of the dataset was artificially balanced by the publisher.) Handling this imbalance is part of the preprocessing in Section 3.

In [ ]:
import kagglehub

base_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset/versions/1")

# Locate the folder that contains Training/ and Testing/
DATASET_PATH = base_path
for root, dirs, files in os.walk(base_path):
    if "Training" in dirs and "Testing" in dirs:
        DATASET_PATH = root
        break

TRAIN_DIR = os.path.join(DATASET_PATH, "Training")
TEST_DIR = os.path.join(DATASET_PATH, "Testing")
print("Dataset path:", DATASET_PATH)
print("Training classes:", sorted(os.listdir(TRAIN_DIR)))
print("Testing classes:", sorted(os.listdir(TEST_DIR)))

### 2.2 Load image paths and labels

Each class is a subfolder. The images are listed in a table (one row per image) with their class label and whether they come from the Training or Testing folder. The Testing folder is kept untouched as the final test set.

In [ ]:
CLASS_NAMES = sorted(d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d)))
NUM_CLASSES = len(CLASS_NAMES)
CLASS_INDICES = {name: i for i, name in enumerate(CLASS_NAMES)}

def load_split(split_dir, split_name):
    rows = []
    for label in CLASS_NAMES:
        folder = os.path.join(split_dir, label)
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                rows.append({"image_path": os.path.join(folder, fname), "label": label, "split": split_name})
    return pd.DataFrame(rows)

train_full_df = load_split(TRAIN_DIR, "Training")
test_df = load_split(TEST_DIR, "Testing")
all_df = pd.concat([train_full_df, test_df], ignore_index=True)

print("Classes:", CLASS_INDICES)
print(f"Training images: {len(train_full_df)} | Testing images: {len(test_df)}")
all_df.head()

### 2.3 Exploratory data analysis — class distribution

The class counts show how imbalanced the dataset is. Glioma, the clinically most serious class, is among the smaller classes.

In [ ]:
pd.crosstab(all_df["label"], all_df["split"], margins=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=all_df, x="label", hue="split", order=CLASS_NAMES, palette="viridis", ax=axes[0])
axes[0].set_title("Images per class and split", fontweight="bold")
axes[0].set_xlabel("Tumor type")
for p in axes[0].patches:
    if p.get_height() > 0:
        axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                         ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

counts = train_full_df["label"].value_counts().reindex(CLASS_NAMES)
axes[1].pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("viridis", NUM_CLASSES), wedgeprops={"edgecolor": "black"})
axes[1].set_title("Training set class share", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Imbalance ratio (largest / smallest training class): {counts.max() / counts.min():.2f}")

### 2.4 Sample images and image sizes

The images come from three different collections, so their sizes vary. All images are resized to 224 × 224 during preprocessing.

In [ ]:
num_images = 5
plt.figure(figsize=(15, 12))
for i, label in enumerate(CLASS_NAMES):
    paths = train_full_df[train_full_df["label"] == label]["image_path"].iloc[:num_images]
    for j, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        plt.subplot(NUM_CLASSES, num_images, i * num_images + j + 1)
        plt.imshow(img)
        plt.title(f"{label}\n{img.size[0]}x{img.size[1]}", fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()

sizes = pd.DataFrame([Image.open(p).size for p in all_df["image_path"].sample(min(500, len(all_df)), random_state=SEED)],
                     columns=["width", "height"])
print("Image sizes (sample of 500):")
print(sizes.describe().loc[["min", "mean", "max"]].round(0))

## 3. Preprocessing

### 3.1 Duplicate and leakage check

Version 1 merges three public collections, so some images appear more than once — including identical images in both Training and Testing. If a test image is also in the training data, test accuracy is inflated. Exact duplicates are found by hashing the file contents (MD5):
- training images that also appear in Testing are **removed from training** (the test set stays as published);
- repeated images inside Training are kept **only once**.

In [ ]:
def file_md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

train_full_df["md5"] = train_full_df["image_path"].map(file_md5)
test_df["md5"] = test_df["image_path"].map(file_md5)

leak_mask = train_full_df["md5"].isin(set(test_df["md5"]))
conflicts = pd.concat([train_full_df, test_df]).groupby("md5")["label"].nunique()

print(f"Training images also in Testing (leakage): {leak_mask.sum()}")
print(f"Duplicates inside Training:                {train_full_df['md5'].duplicated().sum()}")
print(f"Duplicates inside Testing:                 {test_df['md5'].duplicated().sum()}")
print(f"Identical images with different labels:    {(conflicts > 1).sum()}")

train_clean_df = (train_full_df[~leak_mask]
                  .drop_duplicates(subset="md5")
                  .reset_index(drop=True))
print(f"\nTraining images: {len(train_full_df)} -> {len(train_clean_df)} after cleaning")

### 3.2 Train / validation / test split

- **Validation (15%)** is carved out of the cleaned Training data, stratified so every class keeps its proportion. It is used for early stopping and for choosing between configurations (S1 vs. S2, learning rates, default vs. tuned CNN).
- **Test** is the publisher's Testing folder, used only for the final results.

In [ ]:
train_df, val_df = train_test_split(
    train_clean_df, test_size=VAL_FRACTION, stratify=train_clean_df["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

split_counts = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).reindex(CLASS_NAMES)
split_counts.loc["TOTAL"] = split_counts.sum()
split_counts

### 3.3 Handling class imbalance — augmentation-based oversampling

Real clinical datasets are rarely balanced, so the imbalance is handled in preprocessing rather than by choosing a balanced dataset.

**Method:** in the **training split only**, images from the smaller classes are repeated until every class has as many images as the largest class. The repeats are **not identical copies**: data augmentation (Section 3.4) is applied randomly every time an image is loaded, so each repeat is a different rotated, shifted, zoomed, flipped or brightened version of the original.

**Why only after the split:** oversampling before splitting would place copies of the same image in both training and validation/test, which inflates the scores. The validation and test sets are never oversampled or augmented.

In [ ]:
def oversample(df, seed=SEED):
    target = df["label"].value_counts().max()
    parts = []
    for label, group in df.groupby("label"):
        extra = group.sample(n=target - len(group), replace=True, random_state=seed) if len(group) < target else group.iloc[:0]
        parts.append(pd.concat([group, extra]))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

train_balanced_df = oversample(train_df)

balance = pd.DataFrame({
    "before oversampling": train_df["label"].value_counts(),
    "after oversampling": train_balanced_df["label"].value_counts(),
}).reindex(CLASS_NAMES)
print(balance)

balance.plot(kind="bar", figsize=(9, 4), color=["#94a3b8", "#1f3864"], rot=0,
             title="Training images per class, before and after oversampling")
plt.ylabel("Images")
plt.show()

### 3.4 Data augmentation and image generators

Three generators feed images to every model:

| Generator | Source | Preprocessing |
|---|---|---|
| `train_gen` | oversampled training split | resize to 224×224, scale pixels to [0, 1], **random augmentation**: rotation ±10°, shift ±5%, zoom ±10%, horizontal flip, brightness 0.9–1.1 |
| `val_gen` | validation split | resize, scale to [0, 1] — no augmentation |
| `test_gen` | test split | resize, scale to [0, 1] — no augmentation |

The augmentations are mild on purpose: MRI slices should still look anatomically realistic. Vertical flips are not used because brain scans have a consistent top-bottom orientation. Validation and test are not shuffled, so predictions line up with their labels.

In [ ]:
def make_generators(batch_size=BATCH_SIZE):
    train_aug = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        fill_mode="constant",
        cval=0,
    )
    plain = ImageDataGenerator(rescale=1.0 / 255)
    common = dict(x_col="image_path", y_col="label", classes=CLASS_NAMES, target_size=IMG_SIZE,
                  class_mode="categorical", color_mode="rgb", batch_size=batch_size)
    train = train_aug.flow_from_dataframe(train_balanced_df, shuffle=True, seed=SEED, **common)
    val = plain.flow_from_dataframe(val_df, shuffle=False, **common)
    test = plain.flow_from_dataframe(test_df, shuffle=False, **common)
    return train, val, test

train_gen, val_gen, test_gen = make_generators()
print("Class indices:", train_gen.class_indices)

### 3.5 Check the augmentation

The first row shows one original training image; the second row shows five random augmented versions of it — this is what the oversampled repeats of minority-class images look like to the model. The last cell checks that a batch has the expected shape, pixel range and one-hot labels.

In [ ]:
sample_path = train_df[train_df["label"] == train_df["label"].value_counts().idxmin()]["image_path"].iloc[0]
demo_df = pd.DataFrame({"image_path": [sample_path] * 5, "label": [CLASS_NAMES[0]] * 5})
aug_only = ImageDataGenerator(rotation_range=10, width_shift_range=0.05, height_shift_range=0.05,
                              zoom_range=0.1, horizontal_flip=True, brightness_range=[0.9, 1.1],
                              fill_mode="constant", cval=0, rescale=1.0 / 255)
demo_gen = aug_only.flow_from_dataframe(demo_df, x_col="image_path", y_col="label", classes=CLASS_NAMES,
                                        target_size=IMG_SIZE, batch_size=5, shuffle=False, seed=SEED)
aug_batch, _ = next(demo_gen)

plt.figure(figsize=(15, 6))
plt.subplot(2, 5, 3)
plt.imshow(Image.open(sample_path).convert("RGB").resize(IMG_SIZE))
plt.title("Original")
plt.axis("off")
for i in range(5):
    plt.subplot(2, 5, 6 + i)
    plt.imshow(aug_batch[i])
    plt.title(f"Augmented #{i + 1}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
x_batch, y_batch = next(train_gen)
print("Batch shape:", x_batch.shape, "| Labels shape:", y_batch.shape,
      "| Pixel range:", round(float(x_batch.min()), 3), "to", round(float(x_batch.max()), 3))
train_gen.reset()

## 4. Evaluation helpers

These helper functions are used **after** each model has been trained, so that every model is measured and recorded in exactly the same way. (Model creation and training are written out separately for every model in Sections 5 and 6.)

| Function | What it does |
|---|---|
| `plot_training` | plots training vs. validation accuracy and loss for one training run |
| `predict_labels` | runs a model on a generator and returns the true and predicted classes |
| `cpu_latency_ms` | measures how long the model takes to classify one image on a CPU |
| `evaluate_and_record` | evaluates a model on the validation and test sets, saves the model to Drive and adds one row to `results.csv` |
| `load_results`, `show_results` | read `results.csv` and show the rows for one model |

Columns recorded in `results.csv`:

| Column | Meaning |
|---|---|
| `val_macro_f1` | used to choose the best configuration (S1 vs. S2, learning rate, default vs. tuned) |
| `test_accuracy`, `test_macro_f1` | final performance on the test set |
| `recall_<class>` | test recall for each class — `recall_glioma` matters most clinically |
| `params_M`, `size_MB` | model size |
| `cpu_latency_ms` | time to classify one image on a CPU (relevant for deployment) |

In [ ]:
def plot_training(history, title):
    plt.figure(figsize=(12, 4))
    for i, metric in enumerate(["accuracy", "loss"]):
        plt.subplot(1, 2, i + 1)
        plt.plot(history.history[metric], label=f"train {metric}")
        plt.plot(history.history[f"val_{metric}"], label=f"validation {metric}")
        plt.title(f"{title} — {metric}")
        plt.xlabel("epoch")
        plt.legend()
    plt.tight_layout()
    plt.show()


def predict_labels(model, gen):
    gen.reset()
    probs = model.predict(gen, verbose=0)
    return gen.classes, np.argmax(probs, axis=1)


def cpu_latency_ms(model, runs=30):
    """Median time to classify one image on the CPU (copies the model to the CPU first)."""
    with tf.device("/CPU:0"):
        cpu_model = tf.keras.models.clone_model(model)
        cpu_model.set_weights(model.get_weights())
        x = tf.random.uniform((1,) + IMG_SHAPE)
        for _ in range(3):
            cpu_model(x, training=False)
        times = []
        for _ in range(runs):
            start = time.perf_counter()
            cpu_model(x, training=False)
            times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))


def load_results():
    return pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()


def evaluate_and_record(model, run_name, model_name, track, config, history=None, train_time=None, group="main"):
    """Evaluates on validation + test, saves the model and adds a row to results.csv."""
    model_path = os.path.join(MODEL_DIR, run_name.replace(" | ", "_").replace(" ", "_") + ".keras")
    model.save(model_path)

    y_val, p_val = predict_labels(model, val_gen)
    y_test, p_test = predict_labels(model, test_gen)
    per_class_recall = recall_score(y_test, p_test, labels=range(NUM_CLASSES), average=None, zero_division=0)

    row = {
        "run": run_name, "model": model_name, "track": track, "config": config, "group": group,
        "val_accuracy": accuracy_score(y_val, p_val),
        "val_macro_f1": f1_score(y_val, p_val, average="macro"),
        "test_accuracy": accuracy_score(y_test, p_test),
        "test_macro_f1": f1_score(y_test, p_test, average="macro"),
        **{f"recall_{c}": r for c, r in zip(CLASS_NAMES, per_class_recall)},
        "params_M": round(model.count_params() / 1e6, 3),
        "size_MB": round(os.path.getsize(model_path) / 1e6, 1),
        "cpu_latency_ms": round(cpu_latency_ms(model), 1),
        "epochs": len(history.history["loss"]) if history is not None else None,
        "train_min": round(train_time / 60, 1) if train_time else None,
        "confusion_matrix": json.dumps(confusion_matrix(y_test, p_test, labels=range(NUM_CLASSES)).tolist()),
        "model_path": model_path,
    }

    results = load_results()
    if len(results):
        results = results[results["run"] != run_name]      # re-running a step replaces its old row
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    results.to_csv(RESULTS_CSV, index=False)

    print(classification_report(y_test, p_test, target_names=CLASS_NAMES, digits=3))
    summary = {k: row[k] for k in ["val_macro_f1", "test_accuracy", "test_macro_f1", "recall_glioma",
                                    "params_M", "cpu_latency_ms"] if k in row}
    print(f"Recorded '{run_name}':", {k: round(v, 4) if isinstance(v, float) else v for k, v in summary.items()})
    return row


def show_results(model_name):
    results = load_results()
    cols = ["run", "val_macro_f1", "test_accuracy", "test_macro_f1", "recall_glioma", "params_M", "cpu_latency_ms", "epochs"]
    return results[results["model"] == model_name][[c for c in cols if c in results.columns]]

## 5. Track A — Fine-tuning pretrained models

Each pretrained model is used in the same two stages:

| Stage | What is trained | What stays frozen | Learning rate |
|---|---|---|---|
| **S1 — Feature extraction** | only the new classification head | the whole pretrained backbone | 1e-3 |
| **S2 — Fine-tuning** | the head **and** the top 20 backbone layers | the rest of the backbone, and all BatchNorm layers | 1e-5 |

S2 continues from the model trained in S1: the head has already learned sensible weights, so the pretrained layers can then be adjusted gently with a very small learning rate without being damaged. BatchNormalization layers stay frozen, because updating their statistics with small batches would disturb the pretrained features.

**The new classification head** (the same for all three models): global pooling → Dense(256, ReLU) → Dropout(0.4) → Dense(4, softmax).

**Input scaling.** The generators give pixels in the range [0, 1], but each backbone expects the range it was trained with on ImageNet. A `Rescaling` layer at the start of the model converts it:

| Model | Expected input | Rescaling layer |
|---|---|---|
| MobileNetV2 | [-1, 1] | `Rescaling(2.0, offset=-1.0)` |
| EfficientNetB0 | [0, 255] | `Rescaling(255.0)` (EfficientNet normalises internally) |
| Xception | [-1, 1] | `Rescaling(2.0, offset=-1.0)` |

### 5.1 MobileNetV2

A lightweight pretrained network (≈2.3M backbone parameters) built from depth-wise separable convolutions and inverted residual blocks. It is the pretrained model closest in size to Fibonacci-Net, so it is the most direct efficiency comparison.

#### Step 1 — Load the pretrained backbone

`include_top=False` removes the original ImageNet classifier (1,000 classes); we keep only the feature-extracting layers. `trainable = False` freezes all of them for S1.

In [ ]:
mobilenet_base = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
mobilenet_base.trainable = False

print("Backbone:", mobilenet_base.name)
print("Number of backbone layers:", len(mobilenet_base.layers))

#### Step 2 — Add the input scaling and the new classification head

`training=False` keeps the backbone's BatchNorm layers in inference mode, also later during fine-tuning.

In [ ]:
inputs = Input(shape=IMG_SHAPE)
x = layers.Rescaling(2.0, offset=-1.0)(inputs)          # [0, 1] -> [-1, 1]
x = mobilenet_base(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

mobilenet_model = Model(inputs, outputs, name="mobilenetv2")
mobilenet_model.summary()

#### Step 3 — S1: train the new head (backbone frozen)

Only the new head is trained, with learning rate 1e-3, for up to 15 epochs. **Early stopping** ends training when the validation loss stops improving for 4 epochs and restores the best weights; **ReduceLROnPlateau** halves the learning rate when progress stalls.

In [ ]:
set_seeds()
mobilenet_model.compile(optimizer=Adam(learning_rate=1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_mobilenet_s1 = mobilenet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_S1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ],
)
mobilenet_s1_time = time.time() - start

#### Step 4 — S1: evaluate and record

In [ ]:
plot_training(history_mobilenet_s1, "MobileNetV2 — S1")
evaluate_and_record(mobilenet_model, "MobileNetV2 | S1", "MobileNetV2", "A", "S1", history_mobilenet_s1, mobilenet_s1_time)

#### Step 5 — S2: unfreeze the top 20 backbone layers

The backbone is unlocked, then everything except the last 20 layers is frozen again. BatchNorm layers inside those 20 stay frozen. Keras counts every layer (including BatchNorm, activation and add layers), so the last 20 layers correspond roughly to the backbone's final block.

*If the Colab session restarted after S1, run the two commented lines first to reload the S1 model from Drive.*

In [ ]:
# mobilenet_model = tf.keras.models.load_model(os.path.join(MODEL_DIR, "MobileNetV2_S1.keras"))
# mobilenet_base = mobilenet_model.layers[2]

mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-FT_TOP_LAYERS]:
    layer.trainable = False
for layer in mobilenet_base.layers[-FT_TOP_LAYERS:]:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False

trainable_params = sum(int(np.prod(w.shape)) for w in mobilenet_model.trainable_weights)
print(f"Trainable parameters: {trainable_params:,} of {mobilenet_model.count_params():,}")

#### Step 6 — S2: fine-tune

The model must be **compiled again** after changing which layers are trainable. The learning rate is very small (1e-5) so the pretrained weights are only adjusted to MRI images, not overwritten. Up to 10 epochs, early stopping patience 3.

In [ ]:
set_seeds()
mobilenet_model.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_mobilenet_s2 = mobilenet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FT,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ],
)
mobilenet_s2_time = time.time() - start

#### Step 7 — S2: evaluate and record

In [ ]:
plot_training(history_mobilenet_s2, "MobileNetV2 — S2")
evaluate_and_record(mobilenet_model, "MobileNetV2 | S2", "MobileNetV2", "A", "S2", history_mobilenet_s2, mobilenet_s2_time)

#### Step 8 — MobileNetV2: S1 vs. S2

Validation macro-F1 decides which stage is better for MobileNetV2; that one goes into the final comparison.

In [ ]:
show_results("MobileNetV2")

### 5.2 EfficientNetB0

A modern pretrained network (≈4.0M backbone parameters) that scales depth, width and input resolution together. It usually gives the best accuracy for its size, making it the strongest accuracy reference among the pretrained models.

#### Step 1 — Load the pretrained backbone

`include_top=False` removes the original ImageNet classifier (1,000 classes); we keep only the feature-extracting layers. `trainable = False` freezes all of them for S1.

In [ ]:
efficientnet_base = EfficientNetB0(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
efficientnet_base.trainable = False

print("Backbone:", efficientnet_base.name)
print("Number of backbone layers:", len(efficientnet_base.layers))

#### Step 2 — Add the input scaling and the new classification head

`training=False` keeps the backbone's BatchNorm layers in inference mode, also later during fine-tuning.

In [ ]:
inputs = Input(shape=IMG_SHAPE)
x = layers.Rescaling(255.0)(inputs)                     # [0, 1] -> [0, 255]
x = efficientnet_base(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

efficientnet_model = Model(inputs, outputs, name="efficientnetb0")
efficientnet_model.summary()

#### Step 3 — S1: train the new head (backbone frozen)

Only the new head is trained, with learning rate 1e-3, for up to 15 epochs. **Early stopping** ends training when the validation loss stops improving for 4 epochs and restores the best weights; **ReduceLROnPlateau** halves the learning rate when progress stalls.

In [ ]:
set_seeds()
efficientnet_model.compile(optimizer=Adam(learning_rate=1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_efficientnet_s1 = efficientnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_S1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ],
)
efficientnet_s1_time = time.time() - start

#### Step 4 — S1: evaluate and record

In [ ]:
plot_training(history_efficientnet_s1, "EfficientNetB0 — S1")
evaluate_and_record(efficientnet_model, "EfficientNetB0 | S1", "EfficientNetB0", "A", "S1", history_efficientnet_s1, efficientnet_s1_time)

#### Step 5 — S2: unfreeze the top 20 backbone layers

The backbone is unlocked, then everything except the last 20 layers is frozen again. BatchNorm layers inside those 20 stay frozen. Keras counts every layer (including BatchNorm, activation and add layers), so the last 20 layers correspond roughly to the backbone's final block.

*If the Colab session restarted after S1, run the two commented lines first to reload the S1 model from Drive.*

In [ ]:
# efficientnet_model = tf.keras.models.load_model(os.path.join(MODEL_DIR, "EfficientNetB0_S1.keras"))
# efficientnet_base = efficientnet_model.layers[2]

efficientnet_base.trainable = True
for layer in efficientnet_base.layers[:-FT_TOP_LAYERS]:
    layer.trainable = False
for layer in efficientnet_base.layers[-FT_TOP_LAYERS:]:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False

trainable_params = sum(int(np.prod(w.shape)) for w in efficientnet_model.trainable_weights)
print(f"Trainable parameters: {trainable_params:,} of {efficientnet_model.count_params():,}")

#### Step 6 — S2: fine-tune

The model must be **compiled again** after changing which layers are trainable. The learning rate is very small (1e-5) so the pretrained weights are only adjusted to MRI images, not overwritten. Up to 10 epochs, early stopping patience 3.

In [ ]:
set_seeds()
efficientnet_model.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_efficientnet_s2 = efficientnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FT,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ],
)
efficientnet_s2_time = time.time() - start

#### Step 7 — S2: evaluate and record

In [ ]:
plot_training(history_efficientnet_s2, "EfficientNetB0 — S2")
evaluate_and_record(efficientnet_model, "EfficientNetB0 | S2", "EfficientNetB0", "A", "S2", history_efficientnet_s2, efficientnet_s2_time)

#### Step 8 — EfficientNetB0: S1 vs. S2

Validation macro-F1 decides which stage is better for EfficientNetB0; that one goes into the final comparison.

In [ ]:
show_results("EfficientNetB0")

### 5.3 Xception

A large pretrained network (≈21M backbone parameters) built entirely from depth-wise separable convolutions. It tests whether a much bigger backbone gives a meaningful accuracy gain. Global **max** pooling is used in its head, as in the original elephant-classification experiments. Xception's default input size is 299 × 299, but 224 × 224 is used here so every model sees exactly the same images.

#### Step 1 — Load the pretrained backbone

`include_top=False` removes the original ImageNet classifier (1,000 classes); we keep only the feature-extracting layers. `trainable = False` freezes all of them for S1.

In [ ]:
xception_base = Xception(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
xception_base.trainable = False

print("Backbone:", xception_base.name)
print("Number of backbone layers:", len(xception_base.layers))

#### Step 2 — Add the input scaling and the new classification head

`training=False` keeps the backbone's BatchNorm layers in inference mode, also later during fine-tuning.

In [ ]:
inputs = Input(shape=IMG_SHAPE)
x = layers.Rescaling(2.0, offset=-1.0)(inputs)          # [0, 1] -> [-1, 1]
x = xception_base(x, training=False)
x = GlobalMaxPooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

xception_model = Model(inputs, outputs, name="xception")
xception_model.summary()

#### Step 3 — S1: train the new head (backbone frozen)

Only the new head is trained, with learning rate 1e-3, for up to 15 epochs. **Early stopping** ends training when the validation loss stops improving for 4 epochs and restores the best weights; **ReduceLROnPlateau** halves the learning rate when progress stalls.

In [ ]:
set_seeds()
xception_model.compile(optimizer=Adam(learning_rate=1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_xception_s1 = xception_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_S1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ],
)
xception_s1_time = time.time() - start

#### Step 4 — S1: evaluate and record

In [ ]:
plot_training(history_xception_s1, "Xception — S1")
evaluate_and_record(xception_model, "Xception | S1", "Xception", "A", "S1", history_xception_s1, xception_s1_time)

#### Step 5 — S2: unfreeze the top 20 backbone layers

The backbone is unlocked, then everything except the last 20 layers is frozen again. BatchNorm layers inside those 20 stay frozen. Keras counts every layer (including BatchNorm, activation and add layers), so the last 20 layers correspond roughly to the backbone's final block.

*If the Colab session restarted after S1, run the two commented lines first to reload the S1 model from Drive.*

In [ ]:
# xception_model = tf.keras.models.load_model(os.path.join(MODEL_DIR, "Xception_S1.keras"))
# xception_base = xception_model.layers[2]

xception_base.trainable = True
for layer in xception_base.layers[:-FT_TOP_LAYERS]:
    layer.trainable = False
for layer in xception_base.layers[-FT_TOP_LAYERS:]:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False

trainable_params = sum(int(np.prod(w.shape)) for w in xception_model.trainable_weights)
print(f"Trainable parameters: {trainable_params:,} of {xception_model.count_params():,}")

#### Step 6 — S2: fine-tune

The model must be **compiled again** after changing which layers are trainable. The learning rate is very small (1e-5) so the pretrained weights are only adjusted to MRI images, not overwritten. Up to 10 epochs, early stopping patience 3.

In [ ]:
set_seeds()
xception_model.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_xception_s2 = xception_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FT,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ],
)
xception_s2_time = time.time() - start

#### Step 7 — S2: evaluate and record

In [ ]:
plot_training(history_xception_s2, "Xception — S2")
evaluate_and_record(xception_model, "Xception | S2", "Xception", "A", "S2", history_xception_s2, xception_s2_time)

#### Step 8 — Xception: S1 vs. S2

Validation macro-F1 decides which stage is better for Xception; that one goes into the final comparison.

In [ ]:
show_results("Xception")

## 6. Track B — Models trained from scratch

These models start from random weights and learn only from the brain MRI training data, so they use a higher learning rate and more epochs than fine-tuning.

### 6.1 Custom CNN — default settings

A plain CNN: four blocks of Conv → BatchNorm → MaxPool (32, 64, 128, 256 filters), then a flattened dense head. It is the simple from-scratch reference: what a standard CNN achieves without pretrained weights or special design.

#### Step 1 — Build the model

In [ ]:
cnn_model = Sequential([
    Input(shape=IMG_SHAPE),

    Conv2D(32, (3, 3), activation="relu"), BatchNormalization(), MaxPooling2D((2, 2)),     # block 1
    Conv2D(64, (3, 3), activation="relu"), BatchNormalization(), MaxPooling2D((2, 2)),     # block 2
    Conv2D(128, (3, 3), activation="relu"), BatchNormalization(), MaxPooling2D((2, 2)),    # block 3
    Conv2D(256, (3, 3), activation="relu"), BatchNormalization(), MaxPooling2D((2, 2)),    # block 4

    Flatten(),
    Dense(256, activation="relu"), Dropout(0.5),
    Dense(128, activation="relu"), Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax"),
], name="custom_cnn")

cnn_model.summary()

#### Step 2 — Train

Learning rate 1e-3, up to 40 epochs, early stopping patience 6.

In [ ]:
set_seeds()
cnn_model.compile(optimizer=Adam(learning_rate=1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_cnn = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ],
)
cnn_time = time.time() - start

#### Step 3 — Evaluate and record

In [ ]:
plot_training(history_cnn, "Custom CNN — default")
evaluate_and_record(cnn_model, "Custom CNN | default", "Custom CNN", "B", "default", history_cnn, cnn_time)

### 6.2 Custom CNN — hyperparameter tuning

#### Step 1 — Install KerasTuner

In [ ]:
!pip install -q -U keras-tuner

#### Step 2 — Define the search space

KerasTuner needs a function that builds one model from a set of hyperparameter values (`hp`). It has the same four-block structure as the default CNN, but these settings are left open for the tuner to choose:

| Hyperparameter | Values searched | Why |
|---|---|---|
| `base_filters` | 16, 32, 48 | network width (filters double in each block) |
| `head` | flatten, global average pooling | the flattened head holds most of the default model's ~9.9M parameters and tends to overfit; pooling is much smaller |
| `dense_units` | 128, 256, 512 | size of the dense head |
| `dropout` | 0.3 – 0.6 | regularisation strength |
| `learning_rate` | 1e-4 – 1e-3 (log scale) | step size of the optimiser |

In [ ]:
import keras_tuner as kt

def build_tunable_cnn(hp):
    base_filters = hp.Choice("base_filters", [16, 32, 48])
    head = hp.Choice("head", ["flatten", "gap"])
    dense_units = hp.Choice("dense_units", [128, 256, 512])
    dropout = hp.Float("dropout", 0.3, 0.6, step=0.1)
    learning_rate = hp.Float("learning_rate", 1e-4, 1e-3, sampling="log")

    model = Sequential(name="custom_cnn_tuned")
    model.add(Input(shape=IMG_SHAPE))
    model.add(Conv2D(base_filters, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(base_filters * 2, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(base_filters * 4, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(base_filters * 8, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))

    if head == "flatten":
        model.add(Flatten())
    else:
        model.add(GlobalAveragePooling2D())
    model.add(Dense(dense_units, activation="relu"))
    model.add(Dropout(dropout))
    model.add(Dense(dense_units // 2, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation="softmax"))

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

#### Step 3 — Run the search

Random search over 10 combinations, keeping the one with the best **validation accuracy** (the test set is not used). Each trial trains for at most 15 epochs with early stopping to keep the search affordable. The search is stored on Google Drive, so if Colab disconnects, re-running this cell continues the search instead of restarting it.

In [ ]:
tuner = kt.RandomSearch(
    build_tunable_cnn,
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=1,
    seed=SEED,
    directory=os.path.join(OUTPUT_DIR, "tuner_dir"),
    project_name="custom_cnn",
)

start = time.time()
tuner.search(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)
print(f"Search took {(time.time() - start) / 60:.1f} min")

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:", best_hp.values)
tuner.results_summary(num_trials=5)

#### Step 4 — Build the best model

`tuner.hypermodel.build(best_hp)` builds a fresh, untrained model with the best settings found.

In [ ]:
cnn_tuned_model = tuner.hypermodel.build(best_hp)
cnn_tuned_model.summary()

#### Step 5 — Train the best model

The search trials were shortened to save time, so the best settings are now trained from scratch with the same full schedule as the default CNN (up to 40 epochs, early stopping patience 6). This makes the default and tuned CNNs directly comparable. The learning rate chosen by the tuner is already set in the model.

In [ ]:
set_seeds()
start = time.time()
history_cnn_tuned = cnn_tuned_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ],
)
cnn_tuned_time = time.time() - start

#### Step 6 — Evaluate and record, then compare default vs. tuned

In [ ]:
plot_training(history_cnn_tuned, "Custom CNN — tuned")
evaluate_and_record(cnn_tuned_model, "Custom CNN | tuned", "Custom CNN", "B", "tuned", history_cnn_tuned, cnn_tuned_time)

In [ ]:
show_results("Custom CNN")

### 6.3 Fibonacci-Net (Roy et al. 2025)

A lightweight CNN (≈0.38M parameters) designed specifically for class-imbalanced brain tumor MRI. Its key features:
- **Fibonacci filter counts** across seven blocks: 21, 34, 55, 89, 144, 233, 377;
- **Avg-2Max pooling**, AveragePooling(x) − 2·MaxPooling(x), which emphasises edges, used inside two **Parallel Concatenation Blocks** — skip connections from block 2 → 4 and from block 3 → 5;
- **depth-wise separable convolutions** in the last two blocks, to keep the parameter count low.

Fibonacci-Net has no pretrained weights, so instead of S1/S2 it is tuned by trying **three learning rates** (1e-3, 3e-4, 1e-4). Each run creates a fresh copy of the same architecture.

#### Step 1 — Custom layers

Keras has no built-in Avg-2Max pooling or this depth-wise separable block, so they are defined as custom layers. They are registered with Keras (`register_keras_serializable` + `get_config`) so the trained model can be saved and loaded again later — for example by the dashboard.

In [ ]:
@tf.keras.utils.register_keras_serializable()
class Avg2MaxPooling(layers.Layer):
    """Avg-2Max pooling: AveragePooling(x) - 2 * MaxPooling(x)."""
    def __init__(self, pool_size=3, strides=2, padding="same", **kwargs):
        super().__init__(**kwargs)
        self.pool_size, self.strides, self.padding = pool_size, strides, padding
        self.avg_pool = layers.AveragePooling2D(pool_size, strides, padding)
        self.max_pool = layers.MaxPooling2D(pool_size, strides, padding)

    def call(self, inputs):
        return self.avg_pool(inputs) - (self.max_pool(inputs) + self.max_pool(inputs))

    def get_config(self):
        config = super().get_config()
        config.update({"pool_size": self.pool_size, "strides": self.strides, "padding": self.padding})
        return config


@tf.keras.utils.register_keras_serializable()
class DepthwiseSeparableConv(layers.Layer):
    """Depth-wise separable convolution + BatchNorm + ReLU."""
    def __init__(self, filters, kernel_size=3, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.filters, self.kernel_size, self.strides = filters, kernel_size, strides

    def build(self, input_shape):
        self.dw = layers.DepthwiseConv2D(self.kernel_size, self.strides, padding="same")
        self.pw = layers.Conv2D(self.filters, 1, strides=1)
        self.bn = layers.BatchNormalization()
        super().build(input_shape)

    def call(self, inputs):
        return tf.nn.relu(self.bn(self.pw(self.dw(inputs))))

    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters, "kernel_size": self.kernel_size, "strides": self.strides})
        return config

#### Step 2 — The Fibonacci-Net architecture

This function builds the network block by block. It is called once for each learning rate below, so each run starts from fresh random weights.

In [ ]:
def create_fibonacci_net():
    inputs = layers.Input(shape=IMG_SHAPE)

    # Block 1 (21 filters)
    x = layers.Conv2D(21, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)                        # 112 x 112

    # Block 2 (34 filters)
    x = layers.Conv2D(34, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x2 = layers.ReLU()(x)                                # kept for skip connection 1
    x = layers.MaxPooling2D(2)(x2)                       # 56 x 56

    # Block 3 (55 filters)
    x = layers.Conv2D(55, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x3 = layers.ReLU()(x)                                # kept for skip connection 2
    x = layers.MaxPooling2D(2)(x3)                       # 28 x 28

    # Parallel Concatenation Block 1: block 2 -> block 4
    pcb1 = layers.Conv2D(24, 3, padding="same")(x2)
    pcb1 = Avg2MaxPooling()(pcb1)
    pcb1 = layers.Conv2D(24, 3, padding="same")(pcb1)
    pcb1 = Avg2MaxPooling()(pcb1)

    # Block 4 (89 filters) + skip connection 1
    x = layers.Conv2D(89, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)                        # 14 x 14
    x = layers.concatenate([x, layers.Resizing(14, 14)(pcb1)])

    # Parallel Concatenation Block 2: block 3 -> block 5
    pcb2 = layers.Conv2D(24, 3, padding="same")(x3)
    pcb2 = Avg2MaxPooling()(pcb2)
    pcb2 = layers.Conv2D(24, 3, padding="same")(pcb2)
    pcb2 = Avg2MaxPooling()(pcb2)

    # Block 5 (144 filters) + skip connection 2
    x = layers.Conv2D(144, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)                        # 7 x 7
    x = layers.concatenate([x, layers.Resizing(7, 7)(pcb2)])

    # Blocks 6 and 7 (233 and 377 filters, depth-wise separable)
    x = DepthwiseSeparableConv(233)(x)
    x = DepthwiseSeparableConv(377)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="fibonacci_net")


create_fibonacci_net().summary()

#### Step 3 — Train with learning rate 1e-3

Up to 40 epochs, early stopping patience 6.

In [ ]:
set_seeds()
fib_1em3_model = create_fibonacci_net()
fib_1em3_model.compile(optimizer=Adam(learning_rate=1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_fib_1em3 = fib_1em3_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ],
)
fib_1em3_time = time.time() - start

#### Step 4 — Evaluate and record (learning rate 1e-3)

In [ ]:
plot_training(history_fib_1em3, "Fibonacci-Net — lr=1e-3")
evaluate_and_record(fib_1em3_model, "Fibonacci-Net | lr=0.001", "Fibonacci-Net", "B", "lr=0.001", history_fib_1em3, fib_1em3_time)

#### Step 5 — Train with learning rate 3e-4

Up to 40 epochs, early stopping patience 6.

In [ ]:
set_seeds()
fib_3em4_model = create_fibonacci_net()
fib_3em4_model.compile(optimizer=Adam(learning_rate=3e-4), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_fib_3em4 = fib_3em4_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ],
)
fib_3em4_time = time.time() - start

#### Step 6 — Evaluate and record (learning rate 3e-4)

In [ ]:
plot_training(history_fib_3em4, "Fibonacci-Net — lr=3e-4")
evaluate_and_record(fib_3em4_model, "Fibonacci-Net | lr=0.0003", "Fibonacci-Net", "B", "lr=0.0003", history_fib_3em4, fib_3em4_time)

#### Step 7 — Train with learning rate 1e-4

Up to 40 epochs, early stopping patience 6.

In [ ]:
set_seeds()
fib_1em4_model = create_fibonacci_net()
fib_1em4_model.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_fib_1em4 = fib_1em4_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ],
)
fib_1em4_time = time.time() - start

#### Step 8 — Evaluate and record (learning rate 1e-4)

In [ ]:
plot_training(history_fib_1em4, "Fibonacci-Net — lr=1e-4")
evaluate_and_record(fib_1em4_model, "Fibonacci-Net | lr=0.0001", "Fibonacci-Net", "B", "lr=0.0001", history_fib_1em4, fib_1em4_time)

#### Step 9 — Choose the best learning rate

The learning rate with the highest **validation** macro-F1 is used for the final comparison and for the ablation in 6.4.

In [ ]:
display(show_results("Fibonacci-Net"))

best_fib_run = load_results().query("model == 'Fibonacci-Net'").sort_values("val_macro_f1", ascending=False).iloc[0]
BEST_FIB_LR = float(best_fib_run["config"].split("=")[1])
print("Best Fibonacci-Net learning rate:", BEST_FIB_LR)

### 6.4 Fibonacci-Net Ablation A — Avg-2Max replaced by max pooling

An **ablation** removes one component to measure how much it contributes. This network is identical to Fibonacci-Net except that the Avg-2Max pooling inside the two skip connections is replaced by ordinary max pooling. It is trained with the best learning rate from 6.3. If the full Fibonacci-Net scores clearly higher — especially on the smaller classes — the Avg-2Max design is contributing to performance.

#### Step 1 — The ablation architecture

Same as Fibonacci-Net; the only changes are the four `MaxPooling2D(3, 2)` layers in the skip connections (marked `# changed`).

In [ ]:
def create_fibonacci_net_ablation_a():
    inputs = layers.Input(shape=IMG_SHAPE)

    # Block 1 (21 filters)
    x = layers.Conv2D(21, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 2 (34 filters)
    x = layers.Conv2D(34, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x2 = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x2)

    # Block 3 (55 filters)
    x = layers.Conv2D(55, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x3 = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x3)

    # Skip connection 1: block 2 -> block 4
    pcb1 = layers.Conv2D(24, 3, padding="same")(x2)
    pcb1 = layers.MaxPooling2D(3, 2, padding="same")(pcb1)          # changed
    pcb1 = layers.Conv2D(24, 3, padding="same")(pcb1)
    pcb1 = layers.MaxPooling2D(3, 2, padding="same")(pcb1)          # changed

    # Block 4 (89 filters) + skip connection 1
    x = layers.Conv2D(89, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.concatenate([x, layers.Resizing(14, 14)(pcb1)])

    # Skip connection 2: block 3 -> block 5
    pcb2 = layers.Conv2D(24, 3, padding="same")(x3)
    pcb2 = layers.MaxPooling2D(3, 2, padding="same")(pcb2)          # changed
    pcb2 = layers.Conv2D(24, 3, padding="same")(pcb2)
    pcb2 = layers.MaxPooling2D(3, 2, padding="same")(pcb2)          # changed

    # Block 5 (144 filters) + skip connection 2
    x = layers.Conv2D(144, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.concatenate([x, layers.Resizing(7, 7)(pcb2)])

    # Blocks 6 and 7 (depth-wise separable)
    x = DepthwiseSeparableConv(233)(x)
    x = DepthwiseSeparableConv(377)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="fibonacci_net_ablation_a")


ablation_model = create_fibonacci_net_ablation_a()
ablation_model.summary()

#### Step 2 — Train with the best learning rate

In [ ]:
set_seeds()
ablation_model.compile(optimizer=Adam(learning_rate=BEST_FIB_LR), loss="categorical_crossentropy", metrics=["accuracy"])

start = time.time()
history_ablation = ablation_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ],
)
ablation_time = time.time() - start

#### Step 3 — Evaluate and record, then compare with the full Fibonacci-Net

In [ ]:
plot_training(history_ablation, "Fibonacci-Net Ablation A")
evaluate_and_record(ablation_model, f"Fibonacci-Net Ablation A | lr={BEST_FIB_LR:g}", "Fibonacci-Net Ablation A", "B",
                    f"lr={BEST_FIB_LR:g}", history_ablation, ablation_time, group="ablation")

In [ ]:
pd.concat([show_results("Fibonacci-Net"), show_results("Fibonacci-Net Ablation A")])

## 7. Comparison and model selection

This section only reads `results.csv`, so it can be run at any time — even in a fresh Colab session after running Section 1.

### 7.1 S1 vs. S2 for each pretrained model (validation macro-F1)

Whether fine-tuning the top 20 layers (S2) improves on training the head alone (S1), for each pretrained backbone. The better stage per backbone (by validation macro-F1) is carried into the final comparison.

In [ ]:
results = load_results()
track_a = results[results["track"] == "A"]
stage_table = track_a.pivot_table(index="model", columns="config", values="val_macro_f1")
stage_table["best stage"] = stage_table.idxmax(axis=1)
stage_table.round(4)

In [ ]:
stage_table.drop(columns="best stage").plot(kind="bar", figsize=(10, 4), rot=0,
                                               color=["#94a3b8", "#1f3864"])
plt.ylabel("Validation macro-F1")
plt.ylim(max(0, stage_table.drop(columns="best stage").min().min() - 0.05), 1.0)
plt.title("S1 vs. S2 (validation macro-F1)", fontweight="bold")
plt.legend(title="Stage")
plt.show()

### 7.2 Final comparison on the test set

One row per model: its best configuration (chosen on validation macro-F1). Ablation runs are listed separately in 7.3.

In [ ]:
main = results[results["group"] == "main"]
best_configs = (main.sort_values("val_macro_f1", ascending=False)
                    .groupby("model", as_index=False).first()
                    .sort_values("test_macro_f1", ascending=False)
                    .reset_index(drop=True))

final_cols = ["model", "track", "config", "test_accuracy", "test_macro_f1", "recall_glioma",
              "params_M", "size_MB", "cpu_latency_ms"]
final_table = best_configs[final_cols]
final_table.to_csv(os.path.join(OUTPUT_DIR, "final_comparison.csv"), index=False)
final_table.style.format({"test_accuracy": "{:.4f}", "test_macro_f1": "{:.4f}", "recall_glioma": "{:.4f}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_df = final_table.melt(id_vars="model", value_vars=["test_accuracy", "test_macro_f1", "recall_glioma"],
                           var_name="metric", value_name="score")
sns.barplot(data=plot_df, x="model", y="score", hue="metric", palette="viridis", ax=axes[0])
axes[0].set_ylim(max(0, plot_df["score"].min() - 0.1), 1.0)
axes[0].set_title("Test performance (best configuration per model)", fontweight="bold")
axes[0].tick_params(axis="x", rotation=20)

colors = {"A": "#1f3864", "B": "#e07b39"}
for _, r in final_table.iterrows():
    axes[1].scatter(r["params_M"], r["test_macro_f1"], s=90, color=colors[r["track"]])
    axes[1].annotate(r["model"], (r["params_M"], r["test_macro_f1"]), xytext=(6, 4), textcoords="offset points")
axes[1].set_xscale("log")
axes[1].set_xlabel("Parameters (millions, log scale)")
axes[1].set_ylabel("Test macro-F1")
axes[1].set_title("Accuracy vs. model size (blue: fine-tuned, orange: from scratch)", fontweight="bold")

plt.tight_layout()
plt.show()

### 7.3 Ablation result

Contribution of Avg-2Max pooling: full Fibonacci-Net (best learning rate) vs. Ablation A (same learning rate, max pooling instead).

In [ ]:
fib_best = results[(results["model"] == "Fibonacci-Net") & (results["config"] == f"lr={BEST_FIB_LR:g}")]
ablation = results[results["model"] == "Fibonacci-Net Ablation A"]
recall_cols = [f"recall_{c}" for c in CLASS_NAMES]
ablation_table = pd.concat([fib_best, ablation])[["model", "config", "test_accuracy", "test_macro_f1"] + recall_cols]
ablation_table.round(4)

### 7.4 Confusion matrices of the two best models

In [ ]:
top2 = final_table.head(2)
fig, axes = plt.subplots(1, len(top2), figsize=(7 * len(top2), 6))
axes = np.atleast_1d(axes)
for ax, (_, r) in zip(axes, top2.iterrows()):
    cm = np.array(json.loads(results[(results["model"] == r["model"]) & (results["config"] == r["config"])]
                             ["confusion_matrix"].iloc[0]))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{r['model']} ({r['config']})", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()

### 7.5 Model selection (decision rule)

The rule was fixed in the project proposal before seeing the results:
1. Find the model with the highest test macro-F1.
2. Every model within **1 percentage point** of it is treated as equally accurate.
3. Among those, select the **smallest** model (fewest parameters, then lowest CPU latency).

This rule chooses Fibonacci-Net only if it is genuinely as accurate as the best pretrained model; otherwise the more accurate model is selected and the accuracy–efficiency trade-off is reported.

In [ ]:
MARGIN = 0.01
best = final_table.iloc[0]
comparable = final_table[final_table["test_macro_f1"] >= best["test_macro_f1"] - MARGIN]
selected = comparable.sort_values(["params_M", "cpu_latency_ms"]).iloc[0]

print(f"Highest test macro-F1: {best['model']} ({best['config']}) = {best['test_macro_f1']:.4f}")
print(f"Models within {MARGIN * 100:.0f} percentage point: {', '.join(comparable['model'])}")
print(f"\nSELECTED: {selected['model']} ({selected['config']})")
print(f"  test macro-F1 {selected['test_macro_f1']:.4f} | glioma recall {selected['recall_glioma']:.4f} | "
      f"{selected['params_M']:.2f}M parameters | {selected['cpu_latency_ms']:.1f} ms per image on CPU")
if selected["model"] != best["model"]:
    print(f"  -> {best['params_M'] / selected['params_M']:.1f}x fewer parameters than {best['model']}, "
          f"with macro-F1 only {(best['test_macro_f1'] - selected['test_macro_f1']) * 100:.2f} points lower.")
elif len(comparable) > 1:
    print(f"  -> it is the most accurate model and also the smallest of the models within {MARGIN * 100:.0f} point.")
elif len(final_table) > 1:
    runner_up = final_table.iloc[1]
    print(f"  -> no other model came within {MARGIN * 100:.0f} point; the next best was "
          f"{runner_up['model']} at {runner_up['test_macro_f1']:.4f}.")

### 7.6 Export the selected model

The selected model and its class names are copied to `OUTPUT_DIR/selected/` for the Grad-CAM notebook and the dashboard. Every model expects an RGB image resized to 224 × 224 with pixels scaled to [0, 1]; any backbone-specific scaling is built into the model.

In [ ]:
import shutil

SELECTED_DIR = os.path.join(OUTPUT_DIR, "selected")
os.makedirs(SELECTED_DIR, exist_ok=True)
selected_path = results[(results["model"] == selected["model"]) & (results["config"] == selected["config"])]["model_path"].iloc[0]
shutil.copy(selected_path, os.path.join(SELECTED_DIR, "brain_tumor_model.keras"))

with open(os.path.join(SELECTED_DIR, "model_info.json"), "w") as f:
    json.dump({
        "model": selected["model"], "config": selected["config"], "class_names": CLASS_NAMES,
        "input_size": list(IMG_SIZE), "input_scaling": "RGB, pixels / 255",
        "test_macro_f1": float(selected["test_macro_f1"]), "test_accuracy": float(selected["test_accuracy"]),
    }, f, indent=2)
print("Exported to", SELECTED_DIR, ":", os.listdir(SELECTED_DIR))